# X2 · Building a scikit-learn compatible estimator

| Session | Time | Previous | Next |
| --- | --- | --- | --- |
| Challenge track, self-study | about 60 to 90 minutes | [X1 · Git, DVC and MLflow](X1_git_dvc_mlflow.ipynb) | [X3 · Design patterns you already use](X3_design_patterns_in_ds_libraries.ipynb) |

During the workshop we wrote our own `fit`/`predict` classes. They work in our own code, but the moment
you hand one to `cross_val_score`, `GridSearchCV` or a `Pipeline`, scikit-learn refuses it. This notebook
shows the (small) contract scikit-learn expects and why each rule exists. Once your class follows it, you
get cross-validation, hyperparameter search, pipelines, cloning and pretty printing for free.

> **Learning goals.** By the end you can:
>
> - explain why scikit-learn needs `get_params`, `set_params` and `clone`, and how `BaseEstimator` provides them from your `__init__` signature
> - write a transformer and a classifier that follow the scikit-learn conventions (`__init__` only stores parameters, learned attributes end in `_`, `fit` returns `self`)
> - validate input with `validate_data` and `check_is_fitted`, and test your estimator with `check_estimator`
> - use your own estimators inside `Pipeline`, `ColumnTransformer` and `GridSearchCV` on the churn data
> - decide when a custom estimator is worth it and when a `FunctionTransformer` or plain function is enough

**Before you start.** Notebooks 02, 04 and 05 (inheritance, abstraction, duck typing). Notebook 06 (code safari) helps, because we will use the same sklearn classes we read there.

This notebook was written against scikit-learn 1.7. The developer API (`validate_data`, `__sklearn_tags__`) changed in 1.6, so older tutorials may show `self._validate_data(...)` or `_more_tags()`. Those are the old way.

**In this notebook**

- Theory 1 · Why scikit-learn needs more than `fit` and `predict`
- Theory 2 · The contract in five rules
- Theory 3 · Validating input and testing with `check_estimator`
- Hands-on A · Repair a broken estimator
- Theory 4 · Your estimator inside pipelines and grid search
- Hands-on B · Your own estimators for the churn data
- Theory 5 · When is a custom estimator worth it?

---

## Setup

Imports and the churn data used throughout this notebook. `X` holds one row per customer, `y` is 1 for customers who churned and 0 otherwise.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.base import (
    BaseEstimator,
    ClassifierMixin,
    OneToOneFeatureMixin,
    TransformerMixin,
    clone,
)
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import NotFittedError
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.estimator_checks import check_estimator
from sklearn.utils.validation import check_is_fitted, validate_data

print("scikit-learn", sklearn.__version__)

DATA_PATH = Path("../data/raw/telco_churn.csv")
raw = pd.read_csv(DATA_PATH)

# Minimal cleaning, as in src/churn/data.py: blank TotalCharges belong to new customers (tenure 0)
customers = raw.set_index("customerID")
# to_numeric turns text into numbers (blanks become NaN), fillna(0.0) replaces the NaNs
customers["TotalCharges"] = pd.to_numeric(customers["TotalCharges"], errors="coerce").fillna(0.0)
X = customers.drop(columns=["Churn"])
y = (customers["Churn"] == "Yes").astype(int)  # True/False per customer, as 1/0
X.shape, round(y.mean(), 3)

---

## Theory 1 · Why scikit-learn needs more than `fit` and `predict`

scikit-learn's tools (`cross_val_score`, `GridSearchCV`, `Pipeline`) work with any estimator, including your
own classes, as long as those classes follow a few conventions. Duck typing suggests that `fit` and `predict`
should be enough. Let's test that with a tiny rule-based classifier that flags every month-to-month customer.

In [ ]:
class ContractRule:
    def fit(self, X, y):
        return self

    def predict(self, X):
        # True/False per customer, turned into a NumPy array of 1/0
        return (X["Contract"] == "Month-to-month").astype(int).to_numpy()


try:
    cross_val_score(ContractRule(), X, y, cv=5, scoring="f1")
except TypeError as exc:
    print(f"TypeError: {exc}")

The error is not about `fit` or `predict`. It is about **cloning**.

`cross_val_score` trains five models, one per fold. It must not reuse your object, because then fold 2 would
start from the state that fold 1 left behind. So it makes fresh, unfitted copies with `sklearn.base.clone`.
`clone` does not deep-copy the object (that would copy fitted state too). Instead it asks the object for its
*hyperparameters* with `get_params()` and builds a new object from them:

```python
new = type(est)(**est.get_params())     # roughly what clone() does
```

`GridSearchCV` goes one step further and calls `set_params(threshold=0.3)` to try each grid point. So the
real contract is: **an estimator must be able to describe its own hyperparameters and be rebuilt from them.**

---

## Theory 2 · The contract in five rules

`BaseEstimator` implements `get_params`, `set_params`, `__repr__` and more, *by reading the signature of
your `__init__`*. That only works if you follow a few conventions:

| Rule | Why |
|---|---|
| `__init__` takes every hyperparameter as an explicit keyword argument (no `*args`, no `**kwargs`) | `get_params` reads the parameter names from the signature |
| `__init__` stores each argument **unchanged** under the **same name**, and does nothing else | `get_params` reads `self.<name>`; `clone` checks that the new object holds exactly the values it passed in |
| Validation of parameters and data happens in `fit`, not in `__init__` | `set_params` bypasses `__init__`, so checks there would be skipped anyway |
| Everything learned from data is stored in attributes ending in `_` (e.g. `classes_`), created in `fit` | `check_is_fitted` looks for such attributes; users can tell hyperparameters from learned state |
| `fit` returns `self`; `transform`/`predict` do not change learned state | enables `Model().fit(X, y).predict(X)` and makes predictions repeatable |

Mixins add the role-specific parts: `TransformerMixin` gives you `fit_transform`, `ClassifierMixin` gives
you `score` (accuracy) and tells sklearn this is a classifier (which, for example, makes `cross_val_score` use
stratified folds). Put mixins **to the left** of `BaseEstimator` so their methods win in the MRO.

In [ ]:
class ContractRuleClassifier(ClassifierMixin, BaseEstimator):
    """Flag customers whose contract type is in `risky_contracts` (the retention team's rule)."""

    def __init__(self, risky_contracts=("Month-to-month",)):
        self.risky_contracts = risky_contracts

    def fit(self, X, y):
        self.classes_ = np.array([0, 1])  # sklearn classifiers expose the labels they know
        return self

    def predict(self, X):
        check_is_fitted(self)  # error if fit() has not run yet (more in Theory 3)
        # isin: True where the contract is one of risky_contracts
        return X["Contract"].isin(self.risky_contracts).astype(int).to_numpy()


rule = ContractRuleClassifier()
print(rule)
print(rule.get_params())
print(clone(rule.set_params(risky_contracts=("Month-to-month", "One year"))))
print("F1 per fold:", cross_val_score(ContractRuleClassifier(), X, y, cv=5, scoring="f1").round(3))

Note what we did *not* write: no `get_params`, no `set_params`, no `__repr__`. `BaseEstimator` derived all
of them from `__init__`. That is inheritance used exactly as intended: a base class providing shared
behaviour based on a convention.

> **Check your understanding**
>
> What happens when scikit-learn clones this class, and why, even though it "only" makes the parameter nicer?
>
> ```python
> class RoundedThreshold(ClassifierMixin, BaseEstimator):
>     def __init__(self, threshold=0.5):
>         self.threshold = round(float(threshold), 1)
> ```
>
> <details><summary><b>Show answer</b></summary>
>
> `clone` builds a new object with `RoundedThreshold(threshold=0.25)` and then checks that
> `new.threshold is` the value it passed. Here it gets `0.2` back, so `clone` raises
> `RuntimeError: Cannot clone object ... as the constructor either does not set or modifies parameter threshold`.
> Worse, `get_params()` would report `0.2`, so a grid search over `[0.25, 0.3]` would silently test the wrong
> values. Store parameters unchanged and do conversions in `fit`.
> </details>

In [ ]:
# Predict the output first, then run this cell
class RoundedThreshold(ClassifierMixin, BaseEstimator):
    def __init__(self, threshold=0.5):
        self.threshold = round(float(threshold), 1)


try:
    clone(RoundedThreshold(threshold=0.25))
    print("clone worked")
except RuntimeError as exc:
    print(f"RuntimeError: {exc}")

---

## Theory 3 · Validating input and testing with `check_estimator`

Our example is a preprocessing step for linear models called **winsorising**: clip each numeric column to quantiles
learned on the training data, so a handful of extreme customers cannot dominate the coefficients. The
quantiles are learned state (they must come from training data only, exactly like the categories of the
encoder in notebook 04).

Two helpers from `sklearn.utils.validation` do the bookkeeping:

- `validate_data(self, X)` in `fit` converts `X` to a numeric array, rejects NaN/inf by default, and records
  `n_features_in_` (and `feature_names_in_` if `X` is a DataFrame).
- `validate_data(self, X, reset=False)` in `transform` checks that new data has the *same* columns as the
  training data. This catches the "columns in a different order" bug for free.
- `check_is_fitted(self)` raises sklearn's `NotFittedError` if no attribute ending in `_` exists yet.

In [ ]:
class QuantileClipper(TransformerMixin, BaseEstimator):
    """Clip each column to the [lower, upper] quantiles seen during fit."""

    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        if not 0 <= self.lower < self.upper <= 1:
            raise ValueError(
                f"Need 0 <= lower < upper <= 1, got lower={self.lower}, upper={self.upper}"
            )
        X = validate_data(self, X)
        # axis=0: one quantile per column
        self.lower_bounds_ = np.quantile(X, self.lower, axis=0)
        self.upper_bounds_ = np.quantile(X, self.upper, axis=0)
        return self

    def transform(self, X):
        check_is_fitted(self)
        X = validate_data(self, X, reset=False)
        # clip: values below the lower bound become the lower bound, above the upper bound the upper bound
        return np.clip(X, self.lower_bounds_, self.upper_bounds_)


numeric = X[["tenure", "MonthlyCharges", "TotalCharges"]]
clipper = QuantileClipper(upper=0.95).fit(numeric)
bounds = zip(clipper.feature_names_in_, clipper.upper_bounds_, strict=True)
print("learned upper bounds:", {col: round(float(b), 1) for col, b in bounds})
print("max before:", numeric.max().round(1).tolist())
print("max after: ", clipper.transform(numeric).max(axis=0).round(1).tolist())

In [ ]:
# The bookkeeping protects us from two classic mistakes
try:
    QuantileClipper().transform(numeric)
except NotFittedError as exc:
    print(f"NotFittedError: {exc}")

try:
    clipper.transform(
        numeric[["MonthlyCharges", "tenure", "TotalCharges"]]
    )  # same columns, wrong order
except ValueError as exc:
    print(f"ValueError: {str(exc).splitlines()[0]}")

### `check_estimator`: the test suite scikit-learn runs on its own estimators

`check_estimator` runs dozens of generic checks: cloning, `get_params` round trips, that `fit` returns
`self`, that `transform` does not change learned state, behaviour on NaN, on a single sample, on pandas input,
pickling, and more. By default it raises on the first failure. With `on_fail=None` and a `callback` we can
collect every result instead.

In [ ]:
def run_checks(estimator):
    results = []
    check_estimator(
        estimator,
        on_fail=None,
        callback=lambda **kw: results.append((kw["check_name"], kw["status"])),
    )
    # value_counts: how often each status ("passed", "failed", ...) occurs
    statuses = pd.Series([status for _, status in results]).value_counts().to_dict()
    problems = [(name, status) for name, status in results if status != "passed"]
    return statuses, problems


statuses, problems = run_checks(QuantileClipper())
print(statuses)
print(problems or "no failed or skipped checks")

All checks pass for `QuantileClipper` with scikit-learn 1.7. That is not luck: we followed the five rules and
used `validate_data`. If you leave out `validate_data`, several checks fail immediately (no
`n_features_in_`, NaN accepted silently, wrong column count accepted). Try it if you are curious.

`check_estimator` feeds plain numeric NumPy arrays. That is why it suits numeric estimators like this one.
An estimator that needs a DataFrame with a `Contract` column cannot pass the generic checks, and that is fine:
you then test it with your own pytest tests (notebook 10).

---

## Hands-on A · Repair a broken estimator

Time to apply the five rules. Below is a small but useful churn baseline written by a colleague who has not
read Theory 2. It works on its own, but breaks as soon as scikit-learn tools touch it.

### Exercise A.1

The class below breaks the contract in **three** places. Find and fix them, then run the check cell.

It is meant to be a tiny but useful baseline: predict churn for customers with short tenure, where "short"
is a learned cutoff, namely the `quantile` of tenure among customers who churned in the training data.

**What your code should do.**

- `TenureCutoffClassifier(quantile=0.5).get_params()` returns `{"quantile": 0.5, "verbose": False}`, and
  `clone` and `set_params(quantile=0.75)` work.
- `fit(X, y)` learns the cutoff (a number of months, about 10 for `quantile=0.5`), stores it as `cutoff_` and
  returns the object itself.
- An invalid `quantile` (outside 0 to 1) raises `ValueError` when `fit` is called, not when the object is created.
- `predict(X)` returns a NumPy array of 0s and 1s, with 1 for customers whose tenure is at most the cutoff.

**pandas and NumPy you need.** The data handling is already written in the starter; you only fix the class design.

- `X["tenure"][y == 1]`: the tenure values of the churners only (`y == 1` is a True/False mask).
- `np.quantile(values, 0.5)`: the value below which 50% of the values lie (the median).
- `(X["tenure"] <= cutoff).astype(int).to_numpy()`: True/False per customer, turned into a NumPy array of 1/0.

**Discuss.** Validation moves from `__init__` to `fit` here, although notebook 07 taught "fail early". Why does scikit-learn accept failing later, and in which of your own (non-scikit-learn) classes would you still validate in `__init__`?

In [ ]:
# Exercise A.1: your code here


class TenureCutoffClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, quantile=0.5, verbose=False):
        if not 0 < quantile < 1:
            raise ValueError("quantile must be in (0, 1)")
        self.q = quantile
        self.verbose = verbose

    def fit(self, X, y):
        y = np.asarray(y)
        self.classes_ = np.array([0, 1])
        self.cutoff = float(np.quantile(X["tenure"][y == 1], self.q))
        if self.verbose:
            print(f"learned cutoff: {self.cutoff} months")

    def predict(self, X):
        check_is_fitted(self)
        return (X["tenure"] <= self.cutoff_).astype(int).to_numpy()

In [ ]:
# Check for exercise A.1: run after solving it
est = TenureCutoffClassifier(quantile=0.5)
assert est.get_params() == {"quantile": 0.5, "verbose": False}, est.get_params()
assert clone(est).get_params() == est.get_params()
assert est.set_params(quantile=0.75).quantile == 0.75
fitted = TenureCutoffClassifier().fit(X, y)
assert fitted is not None, "fit must return self"
assert hasattr(fitted, "cutoff_")
scores = cross_val_score(TenureCutoffClassifier(), X, y, cv=5, scoring="f1")
assert scores.mean() > 0.4, scores
try:
    TenureCutoffClassifier(quantile=2).fit(X, y)
    raise AssertionError("an invalid quantile should be rejected in fit")
except ValueError:
    pass
print(f"F1 per fold: {scores.round(3)}")
print("All checks passed")

<details><summary><b>Show hint</b></summary>

1. Look at what `__init__` stores and under which name. 2. Look at what `fit` returns. 3. Compare the
attribute name `fit` sets with the one `predict` reads, and think about which convention it should follow.
</details>

<details><summary><b>Show solution</b></summary>

```python
class TenureCutoffClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, quantile=0.5, verbose=False):
        self.quantile = quantile          # same name as the argument, stored unchanged
        self.verbose = verbose

    def fit(self, X, y):
        if not 0 < self.quantile < 1:     # validation moved from __init__ to fit
            raise ValueError("quantile must be in (0, 1)")
        y = np.asarray(y)
        self.classes_ = np.array([0, 1])
        self.cutoff_ = float(np.quantile(X["tenure"][y == 1], self.quantile))  # learned: trailing _
        if self.verbose:
            print(f"learned cutoff: {self.cutoff_} months")
        return self                       # fit returns self

    def predict(self, X):
        check_is_fitted(self)
        return (X["tenure"] <= self.cutoff_).astype(int).to_numpy()
```

The three violations: the parameter was stored as `self.q` (so `get_params` fails), `fit` returned `None`,
and the learned cutoff had no trailing underscore. Validating in `__init__` is a fourth smell: it is skipped
by `set_params`, so `GridSearchCV` could still pass an invalid value.
</details>

---

## Theory 4 · Your estimator inside pipelines and grid search

Now `QuantileClipper` composes with everything. Below it sits inside a `Pipeline` inside a
`ColumnTransformer`, and `GridSearchCV` tunes its `upper` parameter through the nested name
`pre__num__clip__upper` (step names joined by double underscores; this is `set_params` at work).

In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_cols = ["Contract", "InternetService", "PaymentMethod", "TechSupport"]

preprocess = ColumnTransformer(
    [
        ("num", Pipeline([("clip", QuantileClipper()), ("scale", StandardScaler())]), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ]
)
model = Pipeline([("pre", preprocess), ("clf", LogisticRegression(max_iter=1000))])

search = GridSearchCV(
    model,
    param_grid={"pre__num__clip__upper": [0.9, 0.95, 0.99, 1.0]},
    scoring="roc_auc",
    cv=5,
)
search.fit(X, y)
# cv_results_ is a dict of lists; as a DataFrame it has one row per grid point
pd.DataFrame(search.cv_results_)[
    ["param_pre__num__clip__upper", "mean_test_score", "std_test_score"]
].round(4)

An honest result: clipping hardly matters here. The Telco data has no extreme outliers (monthly charges range
from about 18 to 119), so the differences between grid points are within the fold-to-fold noise
(`std_test_score`). That is a useful outcome too. The value of making the step sklearn-compatible is that we
could *find this out* with three lines of `GridSearchCV` instead of a hand-written loop.

---

## Hands-on B · Your own estimators for the churn data

With the contract in place, your own classes plug into `GridSearchCV` like any scikit-learn model. That lets you
compare simple business rules with machine learning fairly, using the same folds and the same metric.

### Exercise B.1
> If you finish early, try this one.

Write `GroupRateClassifier(column="Contract", threshold=0.5)`:

- `fit(X, y)` learns the churn rate of each category of `column` (e.g. 43% for month-to-month) and the overall
  churn rate as a fallback for categories never seen in training. Store them as `rates_` (a dict) and `prior_`.
- `predict_proba(X)` returns an array of shape `(n, 2)`: `[1 - rate, rate]` per customer.
- `predict(X)` flags customers whose rate is at least `threshold`.

Then let `GridSearchCV` choose both the column (`Contract`, `InternetService`, `PaymentMethod`) and the
threshold (`0.2, 0.3, 0.4, 0.5`) by F1. Before running it: with `threshold=0.5`, what F1 do you expect for
`column="Contract"`, given the churn rates by contract?

**What your code should do.**

- After `fit(X, y)`: `rates_` is a dict from category to churn rate, e.g.
  `{"Month-to-month": 0.43, "One year": 0.11, "Two year": 0.03}`, and `prior_` is the overall churn rate (about 0.265).
- `predict_proba(X)` returns a NumPy array with one row per customer and two columns, `[1 - rate, rate]`. A
  category that was not seen in training uses `prior_` as its rate.
- `predict(X)` returns a NumPy array of 0s and 1s: 1 where the rate is at least `threshold`.
- `predict` before `fit` raises scikit-learn's `NotFittedError` (use `check_is_fitted`).
- `group_search` is a `GridSearchCV` that you have already called `.fit(X, y)` on.

**pandas and NumPy you need.**

- `pd.Series(np.asarray(y), index=X.index)`: `y` as a Series with the same row labels as `X`, so the two line up.
- `y.groupby(X["Contract"]).mean()`: one churn rate per contract type (a Series indexed by contract);
  `.to_dict()` turns it into a dict.
- `X["Contract"].map(rates)`: replaces each category by its value in the dict `rates`; unknown categories
  become `NaN`. `.fillna(prior)` replaces those `NaN`s.
- `.to_numpy(dtype=float)`: a pandas Series as a NumPy array of floats.
- `np.column_stack([a, b])`: puts two 1-D arrays side by side as the two columns of one array.
- `proba[:, 1]`: the second column of a 2-D array (the churn probabilities).

**Discuss.** How close does a one-column baseline like this get to the full model? If a complex model beat it by only a few points of F1, what would you tell the retention team, and which model would you deploy?

In [ ]:
# Exercise B.1: your code here


class GroupRateClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, column="Contract", threshold=0.5):
        raise NotImplementedError("Your code here")

    def fit(self, X, y):
        raise NotImplementedError("Your code here")

    def predict_proba(self, X):
        raise NotImplementedError("Your code here")

    def predict(self, X):
        raise NotImplementedError("Your code here")


group_search = None  # replace with a fitted GridSearchCV

In [ ]:
# Check for exercise B.1: run after solving it
grc = GroupRateClassifier().fit(X, y)
assert grc.get_params() == {"column": "Contract", "threshold": 0.5}
assert grc.rates_["Month-to-month"] > 0.4 and grc.rates_["Two year"] < 0.05
proba = grc.predict_proba(X.head(3))
assert proba.shape == (3, 2) and np.allclose(proba.sum(axis=1), 1)
unseen = X.head(1).assign(Contract="Three year")
assert np.isclose(grc.predict_proba(unseen)[0, 1], y.mean()), (
    "unseen category should fall back to prior_"
)
try:
    GroupRateClassifier().predict(X)
    raise AssertionError("predict before fit should raise")
except NotFittedError:
    pass
assert isinstance(group_search, GridSearchCV) and hasattr(group_search, "best_params_")
print("best:", group_search.best_params_, "F1 =", round(group_search.best_score_, 3))
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
class GroupRateClassifier(ClassifierMixin, BaseEstimator):
    """Predict the training churn rate of the customer's group (e.g. their contract type)."""

    def __init__(self, column="Contract", threshold=0.5):
        self.column = column
        self.threshold = threshold

    def fit(self, X, y):
        y = pd.Series(np.asarray(y), index=X.index)
        self.classes_ = np.array([0, 1])
        self.rates_ = y.groupby(X[self.column]).mean().to_dict()
        self.prior_ = float(y.mean())
        return self

    def predict_proba(self, X):
        check_is_fitted(self)
        rate = X[self.column].map(self.rates_).fillna(self.prior_).to_numpy(dtype=float)
        return np.column_stack([1 - rate, rate])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= self.threshold).astype(int)


group_search = GridSearchCV(
    GroupRateClassifier(),
    param_grid={
        "column": ["Contract", "InternetService", "PaymentMethod"],
        "threshold": [0.2, 0.3, 0.4, 0.5],
    },
    scoring="f1",
    cv=5,
).fit(X, y)
```

With `threshold=0.5` and `column="Contract"` the F1 is 0: the riskiest group (month-to-month) churns at
about 43%, below 0.5, so nobody is flagged. The search picks a lower threshold, which is the same lesson as
in notebook 03: the threshold is a business decision, not a fixed 0.5.

The model reaches an F1 of about 0.58 with a single column. Keep such a baseline next to your fancy model:
if the fancy model cannot beat it clearly, something is wrong.
</details>

### Exercise B.2
> **Challenge**

Two extensions for `QuantileClipper`. Write them as a new class `QuantileClipperV2` so the cells above keep working.

1. **Pandas output.** `QuantileClipper().set_output(transform="pandas")` currently raises `AttributeError`.
   `set_output` is only offered when the estimator can name its output columns via `get_feature_names_out`.
   Find the mixin in `sklearn.base` that provides `get_feature_names_out` for transformers whose output columns
   correspond one-to-one to their input columns, and add it.
2. **Missing values.** Real monthly batches arrive with gaps. Make the clipper accept NaN (ignore it when
   learning the quantiles, keep it as NaN in the output) and tell scikit-learn about it, so `check_estimator`
   tests the NaN behaviour instead of expecting an error. Look up `__sklearn_tags__` and `input_tags.allow_nan`
   in the "Developing scikit-learn estimators" guide, and the `ensure_all_finite` argument of `validate_data`.

Then run `run_checks(QuantileClipperV2())`.

**What your code should do.**

- `QuantileClipperV2().set_output(transform="pandas").fit_transform(numeric)` returns a DataFrame with the same
  column names as `numeric`.
- A `NaN` in the input is ignored when learning the quantiles and stays `NaN` in the output.
- `run_checks(QuantileClipperV2())` reports no failed checks.

**pandas and NumPy you need.**

- `np.nanquantile(X, q, axis=0)`: the `q` quantile of every column, ignoring `NaN`.
- `np.clip(X, low, high)`: limits every value to the range `[low, high]` column by column; `NaN` stays `NaN`.

**Discuss.** Accepting NaN means gaps pass silently to the next step. For a monthly scoring batch, should missing values flow through, or stop the pipeline with an error? Who should make that decision?

In [ ]:
# Exercise B.2: your code here


class QuantileClipperV2(TransformerMixin, BaseEstimator):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        raise NotImplementedError("Your code here")

    def transform(self, X):
        raise NotImplementedError("Your code here")

In [ ]:
# Check for exercise B.2: run after solving it
out = QuantileClipperV2().set_output(transform="pandas").fit_transform(numeric)
assert isinstance(out, pd.DataFrame) and out.columns.tolist() == numeric_cols
with_gap = np.array([[1.0], [np.nan], [100.0], [2.0]])
clipped = QuantileClipperV2(lower=0.0, upper=0.5).fit_transform(with_gap).ravel()
assert np.isnan(clipped[1]) and clipped[2] == 2.0, clipped
statuses, problems = run_checks(QuantileClipperV2())
assert not problems, problems
print(statuses)
print("All checks passed")

<details><summary><b>Show hint</b></summary>

The mixin is `OneToOneFeatureMixin`; put it first in the base class list. For NaN: `np.nanquantile`, and
`validate_data(self, X, ensure_all_finite="allow-nan")` in both `fit` and `transform`.
</details>

<details><summary><b>Show solution</b></summary>

```python
class QuantileClipperV2(OneToOneFeatureMixin, TransformerMixin, BaseEstimator):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def __sklearn_tags__(self):
        tags = super().__sklearn_tags__()
        tags.input_tags.allow_nan = True
        return tags

    def fit(self, X, y=None):
        if not 0 <= self.lower < self.upper <= 1:
            raise ValueError(f"Need 0 <= lower < upper <= 1, got {self.lower}, {self.upper}")
        X = validate_data(self, X, ensure_all_finite="allow-nan")
        self.lower_bounds_ = np.nanquantile(X, self.lower, axis=0)
        self.upper_bounds_ = np.nanquantile(X, self.upper, axis=0)
        return self

    def transform(self, X):
        check_is_fitted(self)
        X = validate_data(self, X, reset=False, ensure_all_finite="allow-nan")
        return np.clip(X, self.lower_bounds_, self.upper_bounds_)  # NaN stays NaN
```

Why tags instead of just accepting NaN? `check_estimator` reads the tags to decide which checks apply. Without
the tag it would feed NaN and expect a clear error; with it, it checks that NaN is handled. Tags are how an
estimator *declares* its capabilities, so tools (and other developers) do not have to guess.
</details>

---

## Theory 5 · When is a custom estimator worth it?

| Situation | Better choice |
|---|---|
| A stateless transformation (log, ratio of two columns) | `FunctionTransformer(np.log1p)` or a column in your cleaning code |
| Something sklearn already has (scaling, one-hot, imputation) | the sklearn class: it is tested on far more edge cases |
| A step that **learns** from data and must be tuned, cross-validated or shipped inside a pipeline | a custom estimator following the contract |
| A domain rule you want to compare fairly against ML models | a small custom classifier, as in Exercise B.1 |

Our reference package `src/churn` deliberately does *not* follow the sklearn contract: its `ChurnModel` takes a
config object and wraps a sklearn estimator (composition). That is a valid design for an application. If you
wanted `GridSearchCV` over `ChurnModel` itself, you would refactor it along the rules in Theory 2.

---

## Summary

- scikit-learn tools work with any object that follows a small contract, and the heart of the contract is
  "can describe its hyperparameters and be rebuilt from them" (`get_params`, `set_params`, `clone`).
- `BaseEstimator` implements that contract by reading your `__init__` signature, so `__init__` must store
  every argument unchanged under the same name and do nothing else.
- Learned state goes into attributes ending in `_`, created in `fit`; `fit` returns `self`.
- `validate_data` and `check_is_fitted` give you input checks, `n_features_in_` and column-order protection for free.
- `check_estimator` is a ready-made test suite for numeric estimators; tags declare capabilities such as NaN support.

**Common pitfalls:** validating or converting parameters in `__init__`; forgetting `return self`; storing
learned values without the underscore; mixins listed after `BaseEstimator`; copying old tutorials that use
`self._validate_data` or `_more_tags` (removed in recent versions).

## Going further

- Read `sklearn/preprocessing/_data.py` (`StandardScaler`) with today's rules in mind: you will recognise every one.

---

## Sources and further reading

**Sources.** This notebook is based on:

- [scikit-learn: Developing scikit-learn estimators](https://scikit-learn.org/stable/developers/develop.html), the official rules this notebook follows.
- [`check_estimator`](https://scikit-learn.org/stable/modules/generated/sklearn.utils.estimator_checks.check_estimator.html) and the [`set_output` API example](https://scikit-learn.org/stable/auto_examples/miscellaneous/plot_set_output.html).
- [scikit-learn glossary](https://scikit-learn.org/stable/glossary.html).

**For self-study.** Where to go next if you want to learn more:

- [Buitinck et al. (2013), *API design for machine learning software: experiences from the scikit-learn project*](https://arxiv.org/abs/1309.0238) (paper, intermediate): why the API looks the way it does.
- [scikit-learn-contrib project template](https://github.com/scikit-learn-contrib/project-template) (repository, advanced): a ready-made package skeleton for your own estimators.
- Read `sklearn/preprocessing/_data.py` (`StandardScaler`) in the [scikit-learn source](https://github.com/scikit-learn/scikit-learn/tree/main/sklearn) with today's rules in mind.

---

[← X1 · Git, DVC and MLflow](X1_git_dvc_mlflow.ipynb) &nbsp;|&nbsp; [Course overview](../README.md) &nbsp;|&nbsp; [X3 · Design patterns you already use →](X3_design_patterns_in_ds_libraries.ipynb)